# Lab 12: Static games in code
**MST 0441: Consumers, Trade and Business Strategy**
*Second in-person block, session 12. About 45 minutes.*

You will write a Nash-equilibrium finder in about eight lines, run it on the
games from A12, and then give the coordination game and the audit game to an
assistant as a player, recording what it chooses.

### How this lab works

1. Run `Runtime -> Run all` first. The notebook runs as it stands. Read the
   output, then return to the top.
2. Do the cells marked YOUR TURN. Each asks for a number, a line, or a short
   function. The cells are independent; a wrong answer in one does not affect
   the others.
3. Each YOUR TURN ends with a `check(...)` that reports whether your answer
   matches. Nothing raises an error.
4. The last section, *Work with your assistant*, calls your model from code
   through `ask_model()`. One-time setup: the key guide on It's Learning. No key?
   Every prompt is a plain string you can copy into a chat window instead;
   paste the reply where marked. Test each reply in code before accepting it.

Nothing to install. `numpy`, `matplotlib` and `requests` are preinstalled in
Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

def check(name, got, want, tol=1e-9):
    """Friendly checker: prints, never raises."""
    if got is None:
        print(f"  ..  {name}: not filled in yet")
        return False
    try:
        if isinstance(want, (list, tuple, np.ndarray)):
            good = np.allclose(np.asarray(got, dtype=float),
                               np.asarray(want, dtype=float), atol=tol)
        elif isinstance(want, str):
            good = str(got).strip().lower() == want.strip().lower()
        elif isinstance(want, bool):
            good = bool(got) is want
        else:
            good = abs(float(got) - float(want)) < tol
    except Exception as e:
        print(f"  XX  {name}: could not compare ({type(e).__name__}: {e})")
        return False
    print(f"  {'OK ' if good else 'XX '} {name} = {got}" + ("" if good else f"   (expected {want})"))
    return good

print("ready")

## 1. A game, and a solver  (given: just run it)

A game is two payoff matrices. `P1[i, j]` is the row player's payoff when row
plays $i$ and column plays $j$.

This is A12's Problem 1: Kvist and Lundin choose standard $X$, $Y$, or their own
stack $A$.

In [ ]:
labels = ["X", "Y", "A"]
P1 = np.array([[6, 2, 3],
               [2, 4, 3],
               [1, 1, 2]], float)      # Kvist (row)
P2 = np.array([[6, 2, 1],
               [2, 4, 1],
               [3, 3, 2]], float)      # Lundin (column)

def pure_nash(P1, P2, labels):
    """A cell is a Nash equilibrium if neither player wants to move."""
    eq = []
    for i in range(P1.shape[0]):
        for j in range(P1.shape[1]):
            row_ok = P1[i, j] >= P1[:, j].max() - 1e-12
            col_ok = P2[i, j] >= P2[i, :].max() - 1e-12
            if row_ok and col_ok:
                eq.append((labels[i], labels[j], P1[i, j], P2[i, j]))
    return eq

for e in pure_nash(P1, P2, labels):
    print(f"Nash equilibrium: ({e[0]}, {e[1]})  payoffs ({e[2]:g}, {e[3]:g})")

## 2. YOUR TURN: find the strictly dominated strategy

A row is strictly dominated if some **other** row gives a strictly higher payoff
against **every** column. Fill in the comparison.

In [ ]:
def strictly_dominated_rows(P):
    """Return the indices of rows that are strictly dominated."""
    dominated = []
    for i in range(P.shape[0]):
        for k in range(P.shape[0]):
            if i == k:
                continue
            beats_everywhere = None      # <-- YOUR TURN: one boolean expression
            if beats_everywhere is None:
                return None
            if beats_everywhere:
                dominated.append(i)
                break
    return dominated

d = strictly_dominated_rows(P1)
print("dominated rows for Kvist:",
      [labels[i] for i in d] if d is not None else "(not filled in yet)")
check("number of dominated rows", len(d) if d is not None else None, 1)
check("it is A", labels[d[0]] if d else None, "A")

## 3. YOUR TURN: the mixed equilibrium

Delete $A$ and you are left with the $2\times2$ coordination game. The mixing
probability comes from the **opponent's** indifference condition, never from
your own payoffs.

If Kvist plays $X$ with probability $p$, Lundin is indifferent when
$$6p + 2(1-p) = 2p + 4(1-p).$$

In [ ]:
R1 = P1[:2, :2]     # reduced 2x2
R2 = P2[:2, :2]

def mixed_ne_2x2(P_opponent):
    """p = probability the OTHER player puts on their first strategy,
    solved from THIS player's indifference between rows 0 and 1."""
    a, b = P_opponent[0, 0], P_opponent[0, 1]   # opponent's payoffs if it plays col 0
    c, d = P_opponent[1, 0], P_opponent[1, 1]
    # indifference: a*p + b*(1-p) = c*p + d*(1-p)
    p = None                                    # <-- YOUR TURN: solve for p
    return p

p = mixed_ne_2x2(R2.T)      # transpose: put Lundin's payoffs in row form
print("Kvist plays X with probability p =", p)
check("p", p, 1/3)

if p is not None:
    q = p
    payoff = (p*q*R1[0,0] + p*(1-q)*R1[0,1] + (1-p)*q*R1[1,0] + (1-p)*(1-q)*R1[1,1])
    miscoord = p*(1-q) + (1-p)*q
    print(f"expected payoff = {payoff:.4f}  (should be 10/3 = {10/3:.4f})")
    print(f"probability they end up on DIFFERENT standards = {miscoord:.4f}"
          f"  (should be 4/9 = {4/9:.4f})")
    check("mixed payoff", payoff, 10/3)
    check("miscoordination", miscoord, 4/9)

## 4. YOUR TURN: the audit game

A12 Problem 2. Berg AS chooses Evade/Comply; the authority chooses Audit/Not.
There is no pure equilibrium.

In [ ]:
# rows: Evade, Comply ; cols: Audit, Not
B = np.array([[-20.,  30.],
              [  0.,   0.]])        # Berg
S = np.array([[ 10., -30.],
              [ -5.,   0.]])        # Authority

print("pure equilibria:", pure_nash(B, S, ["Evade", "Comply"]))   # expect none

q_audit = mixed_ne_2x2(B)          # Berg indifferent -> pins the AUDIT rate
p_evade = mixed_ne_2x2(S.T)        # Authority indifferent -> pins the EVASION rate
check("P(audit)",  q_audit, 3/5)
check("P(evade)",  p_evade, 1/9)

# Now double the penalty: -20 becomes -40. The authority's payoffs do NOT change.
B2 = B.copy(); B2[0, 0] = -40.0
q2 = mixed_ne_2x2(B2)
p2 = mixed_ne_2x2(S.T)
print(f"\nafter doubling the penalty:  P(audit) {q_audit} -> {q2},"
      f"   P(evade) {p_evade} -> {p2}")
check("P(audit) after", q2, 3/7)
check("P(evade) after", p2, 1/9)

Evasion did not move. The tougher penalty reduced auditing, because the
evasion rate is pinned by the authority's payoffs, and Parliament did not
change those.

---
## An agent that solves games

You have written a Nash finder. Hand it to an agent together with this
session's model, and the interesting step becomes the one before the
algebra: turning a situation described in words into a payoff matrix.

In [ ]:
import json

# Your model, as a function. (The TUTOR notebook has its own ask() that
# talks to the course tutor; this is a different function, and both can
# live in one notebook.) Works with a free Gemini key (Google AI Studio)
# Course route: an OpenRouter key running Mistral. One-time setup: key icon
# in the left sidebar -> add secret OPENROUTER_API_KEY, allow notebook access.
# (A free Gemini key under GOOGLE_API_KEY also works, as a fallback.)
# Never paste a key into a cell.

# ---- WHICH MODEL THIS STUDENT GETS --------------------------------------
# INSTRUCTOR: set EXPERIMENT below. "same" gives everyone MODEL_A.
# "split" sends a random half to MODEL_A and the other half to MODEL_B,
# assigned from the student id, so the same student always lands in the
# same arm however many times they re-run the notebook.
EXPERIMENT = "same"                                  # "same" or "split"
MODEL_A    = "mistralai/mistral-medium-3-5"          # strong at tool calling
MODEL_B    = "mistralai/mistral-small-2603"          # smaller, cheaper
GEMINI_MODEL = "gemini-3.6-flash"                    # fallback route only

ACTIVE_MODEL = MODEL_A

def assign_model(student_id=""):
    """Pick this student's model. Deterministic: same id -> same arm."""
    global ACTIVE_MODEL
    if EXPERIMENT == "split" and student_id.strip():
        import hashlib
        digest = hashlib.sha256(student_id.strip().lower().encode()).hexdigest()
        ACTIVE_MODEL = MODEL_A if int(digest, 16) % 2 == 0 else MODEL_B
    else:
        ACTIVE_MODEL = MODEL_A
    return ACTIVE_MODEL

def _get_secret(name):
    """Colab Secrets first; an environment variable as the fallback, so the
    instructor can test outside Colab. Never a value pasted into a cell."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name)

def _credentials():
    for secret, base, model in [
        ("OPENROUTER_API_KEY",
         "https://openrouter.ai/api/v1",
         ACTIVE_MODEL),
        ("GOOGLE_API_KEY",
         "https://generativelanguage.googleapis.com/v1beta/openai",
         GEMINI_MODEL),
    ]:
        key = _get_secret(secret)
        if key:
            return base, key, model
    return None

def _chat(messages, tools=None):
    """One HTTP call to the model. Returns its reply message, or None."""
    creds = _credentials()
    if creds is None:
        print("(no API key found -- see the key guide on It's Learning)")
        return None
    base, key, model = creds
    payload = {"model": model, "messages": messages}
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    try:
        import requests
        r = requests.post(base + "/chat/completions",
                          headers={"Authorization": "Bearer " + key},
                          json=payload, timeout=90)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]
    except Exception as e:
        print(f"(the call failed: {type(e).__name__}: {e})")
        return None


def ask_model(prompt, system=None):
    """One plain call: text in, text out. No tools, no loop."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    reply = _chat(messages)
    return None if reply is None else reply.get("content")


# ---- THE AGENT ----------------------------------------------------------
# A model on its own only writes text. An agent is a model plus TOOLS plus
# a LOOP. Read Conversation.say() once: it is the whole idea, in 20 lines.

class Conversation:
    """An agent you can talk to. It remembers the exchange, and it can call
    your Python functions in the middle of answering -- or ask you a
    question first and wait for your reply."""

    def __init__(self, tools=None, registry=None, system=None, show=True):
        self.tools, self.registry = tools, registry or {}
        self.show = show
        self.messages = [{"role": "system", "content": system}] if system else []

    def say(self, text, max_steps=6):
        """Say something to the agent. Returns its reply, or None with no key."""
        self.messages.append({"role": "user", "content": text})
        for _ in range(max_steps):
            reply = _chat(self.messages, self.tools)
            if reply is None:
                return None
            self.messages.append(reply)
            calls = reply.get("tool_calls")
            if not calls:                       # no tool wanted: it is answering
                return reply.get("content")
            for call in calls:                  # it asked; WE run the function
                name = call["function"]["name"]
                args = json.loads(call["function"]["arguments"] or "{}")
                result = self.registry[name](**args)
                if self.show:
                    print(f"   [agent ran {name}({args})]")
                self.messages.append({"role": "tool", "tool_call_id": call["id"],
                                      "content": json.dumps(result)})
        return "(gave up: too many steps)"


def run_agent(question, tools, registry, max_steps=6, show=True):
    """One-shot version: ask once, get the answer. A Conversation of length 1."""
    return Conversation(tools, registry, show=show).say(question, max_steps)


# ---- THIS SESSION'S TUTOR ----------------------------------------------
# The rules are the same in every lab. What changes is the MODEL the agent
# is teaching and the PROBLEM SET it can look up -- both handed in below.

TUTOR_RULES = """You are a teaching assistant in a first-year microeconomics
course. Rules you always follow:
1. Explain the METHOD before any numbers, in the order the course teaches it.
2. Never invent numbers. Get them by calling your tools.
3. If a parameter you need has not been given, ASK for it and stop. Do not
   assume a value.
4. When you report a result, say in plain words what it means economically,
   including what a high and a low value of the key parameter would imply.
5. Asked about a problem set question, call get_problem FIRST so you work from
   the real wording. Explain the method and the setup. Do NOT give the final
   numbers: leave those to the student. If they show you an answer, check it
   with your tools and say only whether it is right and which step failed.
6. Asked to test the student, ask ONE question at a time and wait. Say whether
   the reply is right before asking the next. Test understanding rather than
   recall: ask why something holds, or what would change if a parameter moved.
7. Be brief. Six sentences at most."""


def make_tutor(session, model_text, problems, tools, registry, show=True):
    """Build this session's tutor: shared rules, this session's knowledge."""
    def get_problem(problem_id):
        return problems.get(problem_id,
                            "no problem " + str(problem_id) + " in this session")

    reg = dict(registry)
    reg["get_problem"] = get_problem
    tls = list(tools) + [{
        "type": "function",
        "function": {
            "name": "get_problem",
            "description": ("Fetch the exact wording of a problem from THIS "
                            "session's problem set, so you work from the real "
                            "question rather than one you imagined. Valid ids: "
                            + ", ".join(sorted(problems)) + "."),
            "parameters": {"type": "object", "properties": {
                "problem_id": {"type": "string", "enum": sorted(problems)}},
                "required": ["problem_id"]}}}]

    system = (TUTOR_RULES + """

THE MODEL YOU ARE TEACHING IN THIS SESSION:
""" + model_text + """

The student is working through a lab on exactly this material, and has the
problem set open beside them.""")
    return Conversation(tls, reg, system=system, show=show)


print("ask_model() and run_agent() ready")

In [ ]:
#@title Session 12 knowledge (double-click if you want to read it)
SESSION12_MODEL = """Static games of complete information. A game is players,
strategies and payoffs, written as two payoff matrices. A PURE NASH
EQUILIBRIUM is a cell where neither player can gain by deviating alone: the
row player's payoff is the best in its column, and the column player's is the
best in its row. A strategy is STRICTLY DOMINATED if some other strategy pays
strictly more against every opponent strategy; dominated strategies are never
played and can be deleted. When no pure equilibrium exists, players MIX, and
the mixing probability of each player is pinned by the OPPONENT'S
indifference, never by their own payoffs -- this is the single most common
student error. Coordination games have multiple equilibria, which raises
selection: payoff dominance versus risk dominance. In the audit game, raising
the penalty on evasion changes the AUDITOR's behaviour, not the evader's,
because the evasion rate is fixed by the auditor's payoffs."""

PROBLEMS12 = {
    "A12.1": ("Kvist and Lundin simultaneously commit to standard X, standard "
              "Y, or their own stack A. Payoffs (Kvist, Lundin): both X (6,6); "
              "both Y (4,4); both A (2,2); X-Y (2,2); X-A (3,1); Y-X (2,2); "
              "Y-A (3,1); A-X (1,3); A-Y (1,3). Parts cover pure equilibria, "
              "strict dominance, the mixed equilibrium of the reduced 2x2 "
              "game, and which equilibrium to expect."),
    "A12.2": ("The audit game. Berg AS chooses Evade or Comply; the authority "
              "chooses Audit or Not. There is no pure equilibrium. Find the "
              "mixed equilibrium, then analyse what doubling the penalty on "
              "evasion does to the audit rate and to the evasion rate."),
}

In [ ]:
def solve_game(payoffs_row, payoffs_col, labels=None):
    """Find every pure Nash equilibrium of a game given as two matrices."""
    P1, P2 = np.array(payoffs_row, float), np.array(payoffs_col, float)
    lab = labels or [f"s{i+1}" for i in range(P1.shape[0])]
    eq = pure_nash(P1, P2, lab)
    return {"pure_equilibria": [{"row": e[0], "col": e[1],
                                 "payoffs": [e[2], e[3]]} for e in eq],
            "none_found": len(eq) == 0,
            "note": "if none, the equilibrium is in mixed strategies"}

def solve_mixed(opponent_payoffs):
    """Mixing probability from the OPPONENT's indifference condition."""
    p = mixed_ne_2x2(np.array(opponent_payoffs, float))
    return {"probability_on_first_strategy": None if p is None else round(p, 6),
            "note": "comes from the opponent's payoffs, never your own"}

TOOLS12 = [
    {"type": "function", "function": {
        "name": "solve_game",
        "description": ("Find all PURE Nash equilibria of a two-player game. "
                        "Pass the row player's payoff matrix and the column "
                        "player's, each as a list of rows, same shape. Returns "
                        "every equilibrium cell, or none_found if the game has "
                        "only mixed equilibria."),
        "parameters": {"type": "object", "properties": {
            "payoffs_row": {"type": "array", "description": "row player's payoff matrix",
                            "items": {"type": "array", "items": {"type": "number"}}},
            "payoffs_col": {"type": "array", "description": "column player's payoff matrix",
                            "items": {"type": "array", "items": {"type": "number"}}},
            "labels": {"type": "array", "description": "strategy names, in order",
                       "items": {"type": "string"}}},
            "required": ["payoffs_row", "payoffs_col"]}}},
    {"type": "function", "function": {
        "name": "solve_mixed",
        "description": ("Mixed-strategy probability in a 2x2 game. Pass the "
                        "OPPONENT'S payoff matrix: the probability is set by "
                        "making the opponent indifferent, not by your own "
                        "payoffs."),
        "parameters": {"type": "object", "properties": {
            "opponent_payoffs": {"type": "array", "description": "the opponent's 2x2 payoffs",
                                 "items": {"type": "array", "items": {"type": "number"}}}},
            "required": ["opponent_payoffs"]}}},
]

tutor12 = make_tutor(12, SESSION12_MODEL, PROBLEMS12, TOOLS12,
                     {"solve_game": solve_game, "solve_mixed": solve_mixed})
print("session 12 tutor ready")

### A situation in words, not a matrix

This is the step the exam actually tests and the one code cannot do for
you: reading a business situation and writing down the game. Watch whether
the agent builds the right matrix before it solves anything.

In [ ]:
story = """Two Norwegian grocery chains, Rema and Kiwi, each decide on the same
morning whether to launch a loyalty app. Building one costs 3. If both launch,
they split the gains and each ends up with 4. If only one launches, it takes
customers from the other: the launcher gets 7 and the other gets 1. If neither
launches, both keep their current profit of 5. What should Rema do?"""

print(tutor12.say(story + " Build the payoff matrix first, then solve it with "
                  "your tools, then advise Rema in three sentences.")
      or "(no key -- see the key guide)")

In [ ]:
# <-- YOUR TURN: write the matrix yourself and check the agent got it right.
my_matrix = None          # e.g. [[4, 7], [1, 5]] for Rema
print("yours:", my_matrix)
if my_matrix is not None:
    print(solve_game(my_matrix, np.array(my_matrix).T.tolist(),
                     ["launch", "wait"]))

### Then let it teach A12.2 and examine you

In [ ]:
print(tutor12.say("Walk me through the method for A12.2, the audit game, "
                  "without giving me the numbers.") or "(no key)")

In [ ]:
my_reply = ""      # <-- YOUR TURN: after it asks you a question, answer here
print(tutor12.say(my_reply) if my_reply else
      'ask it to test you: tutor12.say("test whether I understood mixing")')

> Assistants explain Nash equilibrium fluently and still often play the
> risk-dominant or cooperative-sounding strategy instead. A short test of that
> gap tells you more than any prompt wording.

---
## Take away

* A pure Nash equilibrium is a two-line check: no profitable row deviation, no
  profitable column deviation.
* Mixing probabilities always come from the **opponent's** indifference.
* In the audit game, punishing one side harder mostly changes the other
  side's behaviour. An assistant asked to play it often misses this.
* A tool call replaces the model's guess with your computation. Which tools
  you offer decides what kind of player you get.